In [ ]:
import numpy as np

def sigmoid(x):
    return 1/(1+np.exp(-x))

def mse(y,ycap):
    return((y-ycap)**2).mean()

def deravative(x):
    return x * (1-x)

def predict(x,w):
    r=x
    for v in w:
        r = sigmoid(r.dot(v))
    return r

def acc_clf(y,ycap):
    r = y == ycap      
    pcnt = r[r==True].size
    n=y.size
    acc=(pcnt/n)*100
    return acc

def sd(x):     
    return (((x-x.mean())**2).sum() / (x.size-1))**0.5

def scale(x):  
    return (x-x.mean())/sd(x)

def scalematrix(x):
    ncol=x.shape[1]
    for i in range(ncol):  
        c = x[:,i]      
        x[:,i] = scale(c) 
    return x

In [ ]:
def train(x,y,w,iter,conv = 0.00000001):
    perr = 0
    j = 0
    W1 = w[0]
    W2 = w[1]
    W3 = w[2]
    W4 = w[3]
    for i in range(iter):
        l1 = sigmoid(x.dot(W1))
        l2 = sigmoid(l1.dot(W2))
        l3 = sigmoid(l2.dot(W3))
        l4 = sigmoid(l3.dot(W4))
        cerr = mse(y,l4)
        if i % 250 == 0:
            print("Current Error for every 250 iters : ",cerr)
        diff = abs(perr - cerr)
        if diff <= conv:
            print("Training Completed at : ",i+1, " iters")
            j = 1
            break
        e4 = y - l4
        delta4 = e4 * deravative(l4)
        e3 = delta4.dot(W4.T)
        delta3 = e3 * deravative(l3)
        e2 = delta3.dot(W3.T)
        delta2 = e2*deravative(l2)
        e1 = delta2.dot(W2.T)
        delta1 = e1 * deravative(l1)
        W1 += x.T.dot(delta1)
        W2 += l1.T.dot(delta2)
        W3 += l2.T.dot(delta3)
        W4 += l3.T.dot(delta4)
        perr = cerr
    if j == 0:
        print("Training not yet completed ")
    return(W1,W2,W3,W4)

In [ ]:
f=open('D:/Ashwath/NS_Trainings/MyStuff/diabetic.txt')
hd=f.readline()
lines=f.readlines()
age=[]
wgt=[]
hgt=[]
dstat=[]
for line in lines:
    w=line.strip().split(',')
    age.append(float(w[1]))
    wgt.append(float(w[2]))
    hgt.append(float(w[-2]))
    dstat.append(float(w[-1]))

x=np.c_[age,wgt,hgt]
Y=np.c_[dstat]  
print(x)    
print(Y)

In [ ]:
print("\nAfter Scaling :")
X = scalematrix(x)
print(X)

In [ ]:
#assigning random weights
np.random.seed(101)
w1 = 2*np.random.random((3,5))-1
w2 = 2*np.random.random((5,7))-1
w3 = 2*np.random.random((7,5))-1
w4 = 2*np.random.random((5,1))-1
print(w1)
print(w2)
print(w3)
print(w4)

In [ ]:
W = [np.array(w1),np.array(w2),np.array(w3),np.array(w4)]
#print(W)
theta = train(X,Y,W,10000)

In [ ]:
ycap = predict(X,theta)

print(np.c_[Y.ravel(),ycap.ravel()])

In [ ]:
ycap[ycap>0.5]=1
ycap[ycap<0.5]=0
print(np.c_[Y.ravel(),ycap.ravel()])

In [ ]:
acc_clf(Y,ycap)

In [ ]:
#prediction for new paitents

a=[]
h=[]
w=[]

new=open('D:/Ashwath/NS_Trainings/MyStuff/newpaitents.txt')
pats = new.readlines()[1:]
#print(pats)
for p in pats:
    wd=p.strip().split(',')
    a.append(float(wd[2]))
    w.append(float(wd[3]))
    h.append(float(wd[4]))
P = np.c_[a,w,h]
P = scalematrix(P)
print(P)

In [ ]:
newpredict = predict(P,theta)
print(newpredict)

In [ ]:
newpredict[newpredict>0.5]=1
newpredict[newpredict<0.5]=0
print(newpredict)

In [ ]:
#writing data to the file
new=open('D:/Ashwath/NS_Trainings/MyStuff/newpaitents_Prediction.txt','w')
for p,d in list(zip(pats,newpredict.ravel())):
    ds = 'Diabetic'
    if d == 1:
        ds = "Non_Diabetic"
    print(p.strip()+','+ds+"\n")
    new.write(p.strip()+','+ds+"\n")
new.close()